# Nettoyage de l'historique des prix du gazole (Île-de-France, 2021-2023)

Issue #26 · fiches ANO-02 à ANO-07 et ANO-14 · sortie : `data/processed/historique_gazole_idf.parquet` (contrat D-11).

Chaque année est nettoyée séparément : l'échelle des prix change en 2021 (ANO-06) et les bornes statistiques (ANO-05, ANO-07) sont propres à chaque année.

In [ ]:
import os

# Remonter jusqu'à la racine du dépôt (là où se trouve src/), sans chemin absolu
while not os.path.isdir("src"):
    os.chdir("..")

import time

import pandas as pd

from src.cleaning import charger_historique, nettoyer_historique

print(os.getcwd())

## 1. Contrôles avant nettoyage, puis nettoyage, année par année

Pour chaque année : couverture, valeurs de `pop` (ANO-14), doublons fonctionnels (ANO-04), prix hors seuils (ANO-02, ANO-03), puis nombre de lignes et de prix manquants après `nettoyer_historique()`.

In [ ]:
annees = [2021, 2022, 2023]
tables = []

for annee in annees:
    debut = time.time()
    brut_annee = charger_historique(annee)
    print(f"===== {annee} =====")
    print("lignes avant nettoyage :", len(brut_annee))
    print("période :", brut_annee["date"].min(), "->", brut_annee["date"].max())

    # ANO-14 : valeur de pop hors domaine (N)
    inconnus = brut_annee[brut_annee["pop"] == "N"]
    print("pop = N :", len(inconnus), "relevés,", inconnus["id_pdv"].nunique(), "stations")

    # ANO-04 : même station au même instant (lignes concernées, pas lignes supprimées)
    doublons = brut_annee.duplicated(subset=["id_pdv", "date"], keep=False).sum()
    print("lignes en doublon fonctionnel :", doublons)

    # ANO-02 et ANO-03 : prix hors seuils métier, en euros (2021 est en millièmes, ANO-06)
    prix_brut = pd.to_numeric(brut_annee["prix"], errors="coerce")
    if annee == 2021:
        prix_brut = prix_brut / 1000
    print("prix < 1 € :", (prix_brut < 1).sum(), "| prix > 3 € :", (prix_brut > 3).sum())

    propre_annee = nettoyer_historique(brut_annee)
    print("lignes après nettoyage :", len(propre_annee))
    print("prix manquants après nettoyage :", propre_annee["prix_gazole"].isna().sum())
    print(propre_annee["prix_gazole"].agg(["min", "median", "max"]).round(3).to_string())
    print("durée :", round(time.time() - debut), "s")

    tables.append(propre_annee)
    del brut_annee  # libère la mémoire avant l'année suivante (ET-05)

**Constats** :
- Couverture : chaque fichier couvre bien son année.
- ANO-06 : prix en millièmes d'euro en 2021, en euros en 2022 et 2023.
- ANO-14 : `pop = N` seulement en 2021 : 47 relevés, 1 station.
- ANO-04 : doublons retirés : 2021 0, 2022 3, 2023 5.
- Prix manquants après nettoyage : 2021 43, 2022 63, 2023 56 (ANO-02, 03, 05, 07).

## 2. Assemblage, contrôles et enregistrement

`pd.concat` assemble les trois années en une seule table (contrat D-11). Il n'est pas vu en cours : c'est la seule fonction ajoutée, justifiée par ce besoin.

In [ ]:
historique = pd.concat(tables, ignore_index=True)

# Contrôles exigés par l'issue
assert not historique.duplicated(subset=["id_station", "date_releve"]).any()
assert historique["prix_gazole"].dropna().between(1, 3).all()

os.makedirs("data/processed", exist_ok=True)
historique.to_parquet("data/processed/historique_gazole_idf.parquet", index=False)

# On relit le fichier pour vérifier que rien n'a été perdu à l'enregistrement
relu = pd.read_parquet("data/processed/historique_gazole_idf.parquet")
assert len(relu) == len(historique)
print(relu.dtypes)

# size = lignes, count = prix valides (les NaN ne sont pas comptés)
print(relu.groupby(relu["date_releve"].dt.year)["prix_gazole"]
      .agg(["size", "count", "min", "median", "max"]))
print(len(relu), "lignes,", round(relu.memory_usage(deep=True).sum() / 1e6, 1), "Mo en mémoire")

## 3. Mémoire (ET-05)

Pic observé dans le Gestionnaire des tâches pendant l'exécution : environ 2,3 Go (mesure manuelle, approximative). Chaque année est chargée puis libérée (`del`), et seul le tableau nettoyé est conservé.